# Tensor Decomposition
*Thomas Dooms & Mitch Crabb*

In tutorial 1, we decomposed the bilinear interaction tensor using eigendecomposition. This gives us interpretable features *per output class*, but there are some issues with this. The most important one is orthogonality; you might have overlapping structures in input space which behave differently. The second is somewhat of a consequence where eigendecompositions often yield "superposed" features; we might hope that a 6 decomposes into a few edge-detectors but that generally doesn't happen. If the model shares structure across classes, eigendecomposition can't see it.

Here we take a different approach. Instead of decomposing per class, we decompose the *full* third-order tensor into a set of shared **neurons**, each with explicit input patterns and output weights. The result is a set of interpretable building blocks that the model combines to classify digits.

### Setup

In [51]:
%load_ext autoreload
%autoreload 2

import torch
import plotly.express as px
from plotly.subplots import make_subplots

from image import Model, MNIST
from image.sparse import Model as Sparse
from kornia.augmentation import RandomGaussianNoise

device = "cpu"
train, test = MNIST(train=True, device=device), MNIST(train=False, device=device)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Training the original model

As in tutorial 1, we train a bilinear model with Gaussian noise augmentation to prevent overfitting on individual pixels.

In [52]:
model = Model.from_config(epochs=20).to(device)
model.fit(train, test, RandomGaussianNoise(std=0.4))

train/loss: 0.149, train/acc: 0.956, val/loss: 0.113, val/acc: 0.967: 100%|██████████| 20/20 [00:07<00:00,  2.54it/s]


,train/loss,train/acc,val/loss,val/acc
0,1.131199,0.686456,0.451672,0.8831
1,0.450729,0.873990,0.324314,0.9154
2,0.389520,0.889261,0.287193,0.9226
3,0.347364,0.900340,0.250061,0.9311
4,0.298244,0.914113,0.208244,0.9401
5,0.260067,0.925276,0.175644,0.9497
6,0.234229,0.934065,0.160026,0.9531
7,0.211315,0.939638,0.146847,0.9579
8,0.196227,0.943326,0.138285,0.9600
9,0.186106,0.946575,0.134037,0.9630


### From eigendecomposition to tensor decomposition

Recall that the bilinear model computes $\text{output}_c = \sum_{i,j} B_{c,i,j}\, x_i\, x_j$, where $B$ is the third-order interaction tensor. Eigendecomposition slices $B$ along the class axis and decomposes each slice independently.

We can instead factorize the *entire* tensor at once into a new bilinear layer:
$$B_{c,i,j} \approx \sum_{r=1}^{R} L_{i,r}\, R_{j,r}\, D_{c,r}$$

Each component $r$ is a **neuron** with three parts:
- $L_r \in \mathbb{R}^{784}$, the left input pattern
- $R_r \in \mathbb{R}^{784}$, the right input pattern  
- $D_r \in \mathbb{R}^{10}$, the output weights over classes

The neuron's activation on input $x$ is $(L_r^\top x)(R_r^\top x)$. Using the identity $ab = \tfrac{1}{4}(a+b)^2 - \tfrac{1}{4}(a-b)^2$, this becomes:
$$\tfrac{1}{4}\big((L_r + R_r)^\top x\big)^2 - \tfrac{1}{4}\big((L_r - R_r)^\top x\big)^2$$

So each neuron naturally decomposes into a **positive pattern** $L_r + R_r$ (matching it increases activation) and a **negative pattern** $L_r - R_r$ (matching it decreases activation). These are directly analogous to eigenvectors with positive and negative eigenvalues.

### Fitting the decomposition

We fit a new bilinear layer by maximizing the cosine similarity between the original interaction tensor and our approximation. The `Sparse` model stores $L$, $R$, and $D$ as learnable parameters and is optimized with Muon.

In [53]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

sparse = Sparse.from_config(rank=64).to(device)

optimizer = torch.optim.Muon(sparse.parameters(), lr=0.02, momentum=0.95)
scheduler = CosineAnnealingLR(optimizer, T_max=200)

torch.set_grad_enabled(True)
for _ in (pbar := tqdm(range(200))):
    loss = 1 - sparse.similarity(model)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    pbar.set_description(f"loss: {loss.item():.4f}")
torch.set_grad_enabled(False)

with torch.no_grad():
    orig_acc = (model(test.x).argmax(-1) == test.y).float().mean()
    sparse_acc = (sparse(test.x).argmax(-1) == test.y).float().mean()
    print(f"Original: {orig_acc:.3f}, Sparse: {sparse_acc:.3f}")

loss: -0.0006: 100%|██████████| 200/200 [00:04<00:00, 42.60it/s]

Original: 0.967, Sparse: 0.967


### Visualizing neurons

Each neuron has a positive pattern ($L_r + R_r$), a negative pattern ($L_r - R_r$), and output weights ($D_r$). The `decompose` method returns these normalized and sorted by importance (the product of the factor norms).

In [54]:
def bar_colors(v):
    return ["#2166ac" if x >= 0 else "#b2182b" for x in v]


def plot_component_grid(models, group_labels, n_comp=8, width_per=112, height_per_group=360):
    """Consistent component visualization for CPD-style decompositions."""
    row_names = ["l+r", "l-r", "logits"]
    n_rows = 3 * len(models)
    vertical_spacing = 0.026 if len(models) > 1 else 0.07

    fig = make_subplots(
        rows=n_rows,
        cols=n_comp,
        vertical_spacing=vertical_spacing,
        horizontal_spacing=0.012,
    )

    for r, m in enumerate(models):
        plus, minus, down, sigma = m.decompose()
        plus, minus, down = plus.cpu(), minus.cpu(), down.cpu()
        for c in range(n_comp):
            params = dict(showscale=False, colorscale="RdBu", zmid=0)
            fig.add_heatmap(z=plus[:, c].view(28, 28).flip(0), **params, row=3*r+1, col=c+1)
            fig.add_heatmap(z=minus[:, c].view(28, 28).flip(0), **params, row=3*r+2, col=c+1)
            vals = down[:, c].tolist()
            fig.add_bar(
                x=list(range(10)),
                y=vals,
                marker_color=bar_colors(vals),
                showlegend=False,
                row=3*r+3,
                col=c+1,
            )

    for row in range(1, n_rows + 1):
        for col in range(1, n_comp + 1):
            if row % 3 == 0:
                fig.update_xaxes(
                    visible=True,
                    tickmode="array",
                    tickvals=list(range(10)),
                    ticktext=[str(i) for i in range(10)],
                    tickfont=dict(size=7),
                    row=row,
                    col=col,
                )
                fig.update_yaxes(
                    visible=True,
                    showticklabels=False,
                    zeroline=True,
                    zerolinecolor="#777",
                    row=row,
                    col=col,
                )
            else:
                fig.update_xaxes(visible=False, row=row, col=col)
                fig.update_yaxes(visible=False, row=row, col=col)

    domain_h = (1 - vertical_spacing * (n_rows - 1)) / n_rows
    for row in range(1, n_rows + 1):
        group = (row - 1) // 3
        kind = row_names[(row - 1) % 3]
        label = f"{group_labels[group]}<br>{kind}" if group_labels[group] else kind
        y = 1 - (row - 1) * (domain_h + vertical_spacing) - domain_h / 2
        fig.add_annotation(
            xref="paper", yref="paper", x=-0.035, y=y,
            text=label, showarrow=False,
            xanchor="right", yanchor="middle",
            align="right", font=dict(size=10),
        )

    fig.update_layout(
        width=n_comp * width_per,
        height=len(models) * height_per_group,
        bargap=0.12,
        margin=dict(l=82, r=14, b=24, t=10),
        template="plotly_white",
    )
    return fig


fig = plot_component_grid([sparse], [""], n_comp=8, height_per_group=400)
fig


In [55]:
px.bar(sigma, template="plotly_white", labels=dict(index="component", value="sigma"))

### Exercises

The goal of this part is to implement a decomposition that is more in line with our requirements. This is where you try to find what kinds of priors and structure work well. The code above provides a skeleton with the essentials: the `Sparse` model, the reconstruction loop, and the visualization. There's no right answer and this is a hard task to get right.

### Approach 1: sparsity prior on the unconstrained CPD

The baseline `Sparse` model fits the interaction tensor almost exactly (cosine sim ≈ 1.0), but each component is a *dense* 28×28 pattern. Most pixels are non-zero, and the components are visually noisy and "superposed." Following the tutorial's hint that *"sparsifying the weights is a good way to find interpretable components"*, we add an L1 penalty on $L$, $R$, and $D$ to the existing cosine-similarity loss:

$$\mathcal{L} = (1 - \cos(B_{\text{pred}}, B_{\text{target}})) + \lambda \cdot (\|L\|_1 + \|R\|_1 + \|D\|_1)$$

This pushes each component to use few input pixels (localised stroke/edge features) and to contribute to few classes (sparse class effects, exposing structure shared across digits). $\lambda$ trades fidelity-to-target against sparsity.

In [56]:
import pandas as pd

def train_sparse_l1(lam, n_steps=200):
    m = Sparse.from_config(rank=64).to(device)
    optimizer = torch.optim.Muon(m.parameters(), lr=0.02, momentum=0.95)
    scheduler = CosineAnnealingLR(optimizer, T_max=n_steps)

    for _ in range(n_steps):
        recon = 1 - m.similarity(model)
        l1 = m.left.abs().mean() + m.right.abs().mean() + m.down.abs().mean()
        loss = recon + lam * l1

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

    with torch.no_grad():
        sim = m.similarity(model).item()
        acc = (m(test.x).argmax(-1) == test.y).float().mean().item()
        all_w = torch.cat([m.left.flatten(), m.right.flatten(), m.down.flatten()]).abs()
        sparsity = (all_w < 0.01).float().mean().item()
    return m, dict(lam=lam, sim=sim, acc=acc, sparsity=sparsity)


lams = [0.0, 0.5, 2.0, 5.0, 10.0]

torch.set_grad_enabled(True)
sparsity_models = []
sparsity_results = []
for lam in (pbar := tqdm(lams)):
    m, r = train_sparse_l1(lam)
    sparsity_models.append(m)
    sparsity_results.append(r)
    pbar.set_description(f"lam={lam}: sim={r['sim']:.3f}, acc={r['acc']:.3f}, sparse={r['sparsity']:.2f}")
torch.set_grad_enabled(False)

df_sparsity = pd.DataFrame(sparsity_results)
print(df_sparsity.round(3).to_string(index=False))

lam=10.0: sim=0.998, acc=0.966, sparse=1.00: 100%|██████████| 5/5 [00:19<00:00,  3.88s/it]

 lam   sim   acc  sparsity
 0.0 1.001 0.967     0.189
 0.5 0.998 0.967     0.452
 2.0 0.999 0.967     0.804
 5.0 0.999 0.967     0.996
10.0 0.998 0.966     0.998


In [57]:
fig = plot_component_grid(
    sparsity_models,
    [f"λ={lam}" for lam in lams],
    n_comp=8,
    height_per_group=360,
)
fig


In [58]:
fig = px.line(
    df_sparsity, x='lam', y=['sim', 'acc', 'sparsity'],
    markers=True, template='plotly_white',
    labels=dict(value='', variable='metric', lam='λ'),
)
fig.update_yaxes(range=[0, 1.02])
fig.update_layout(width=700, height=400)
fig

### Approach 1b: sparsity only on the class effects $D$

The all-weights L1 sweep above shows the model is heavily over-parameterised; components themselves can be compressed without cost. A different question is whether the *class-shared* structure of the model becomes visible when we force each component to contribute to only a small subset of digits. This is the original notebook's explicit motivation: eigendecomposition decomposes per class, so it cannot see structure shared across digits.

We therefore sparsify only $D$ (the $10 \times 64$ class-effect matrix). $L$ and $R$ are left unconstrained. If the model has natural class-sharing, e.g. a "loop" component that contributes to 0, 6, 8, 9, sparsity on $D$ should expose those subsets directly in each component's logits panel.

In [59]:
def train_sparse_l1_D(lam, n_steps=200):
    m = Sparse.from_config(rank=64).to(device)
    optimizer = torch.optim.Muon(m.parameters(), lr=0.02, momentum=0.95)
    scheduler = CosineAnnealingLR(optimizer, T_max=n_steps)

    for _ in range(n_steps):
        recon = 1 - m.similarity(model)
        l1 = m.down.abs().mean()
        loss = recon + lam * l1

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

    with torch.no_grad():
        sim = m.similarity(model).item()
        acc = (m(test.x).argmax(-1) == test.y).float().mean().item()
        # fraction of D entries near-zero relative to per-component max
        d_max = m.down.abs().max(dim=0, keepdim=True).values.clamp(min=1e-8)
        d_sparsity = ((m.down.abs() / d_max) < 0.1).float().mean().item()
    return m, dict(lam=lam, sim=sim, acc=acc, d_sparsity=d_sparsity)


lams_d = [0.0, 1.0, 5.0, 20.0]

torch.set_grad_enabled(True)
d_models = []
d_results = []
for lam in (pbar := tqdm(lams_d)):
    m, r = train_sparse_l1_D(lam)
    d_models.append(m)
    d_results.append(r)
    pbar.set_description(f"lam={lam}: sim={r['sim']:.3f}, acc={r['acc']:.3f}, d_sparse={r['d_sparsity']:.2f}")
torch.set_grad_enabled(False)

df_d = pd.DataFrame(d_results)
print(df_d.round(3).to_string(index=False))

lam=20.0: sim=0.996, acc=0.966, d_sparse=0.71: 100%|██████████| 4/4 [00:15<00:00,  3.92s/it]

 lam   sim   acc  d_sparsity
 0.0 1.001 0.967       0.150
 1.0 0.999 0.967       0.600
 5.0 0.998 0.967       0.694
20.0 0.996 0.966       0.706


In [60]:
fig = plot_component_grid(
    d_models,
    [f"λ={lam}" for lam in lams_d],
    n_comp=8,
    height_per_group=360,
)
fig


### Approach 1c: locality bias on top of L1

Sparsity (Approach 1) reveals which pixels matter, but doesn't say anything about *which pairs* of pixels can interact. The bilinear interaction between two distant pixels (e.g. top-left corner with bottom-right corner) is almost certainly meaningless: digits are local stroke structures, not long-range correlations. This is a real property of MNIST that a generic L1 prior ignores.

We encode this directly as a **locality penalty** on the bilinear interaction. For component $r$, the spatial outer product $|L_r| \otimes |R_r|$ assigns mass to each pixel pair $(i, j)$. Penalise the average grid distance under that mass:

$$
\mathcal{L}_{\text{loc}} = \frac{1}{R} \sum_{r=1}^{R} \frac{\sum_{i,j} D_{i,j}\, |L_{i,r}|\, |R_{j,r}|}{\sum_{i,j} |L_{i,r}|\, |R_{j,r}|},
\qquad D_{i,j} = \|p_i - p_j\|_2 \,/\, \max_{i,j}\|p_i - p_j\|_2
$$

This is the bilinear analogue of nearest-neighbour Ising (something I have spent time working on for my dissertation): long-range pairwise interactions are expensive, local ones are free. Unlike a hard reflection, or translation-equivariant architecture, locality respects an *actual* property of the data. Handwritten digits are spatially coherent, but they are not globally symmetric.

We fix $\lambda_{\ell_1} = 2$ (where Approach 1 already gave clean sparsity at full accuracy) and sweep $\lambda_{\text{loc}}$. The reported `dist` column is the mean per-component pair-distance (small means each component lives on a tight neighbourhood of pixel pairs).

In [61]:
# Pairwise pixel distance on the 28x28 grid, normalized to [0, 1].
P = 28
coords = torch.stack(torch.meshgrid(
    torch.arange(P, dtype=torch.float, device=device),
    torch.arange(P, dtype=torch.float, device=device),
    indexing='ij',
), -1).reshape(-1, 2)
dist = torch.cdist(coords, coords)
dist = dist / dist.max()


def train_sparse_l1_loc(lam_l1, lam_loc, n_steps=200):
    m = Sparse.from_config(rank=64).to(device)
    optimizer = torch.optim.Muon(m.parameters(), lr=0.02, momentum=0.95)
    scheduler = CosineAnnealingLR(optimizer, T_max=n_steps)

    for _ in range(n_steps):
        recon = 1 - m.similarity(model)
        l1 = m.left.abs().mean() + m.right.abs().mean() + m.down.abs().mean()
        Labs, Rabs = m.left.abs(), m.right.abs()
        # Mass-normalised average pixel-pair distance per component.
        num = torch.einsum('iH, ij, jH -> H', Labs, dist, Rabs)
        den = torch.einsum('iH, jH -> H', Labs, Rabs).clamp(min=1e-8)
        locality = (num / den).mean()
        loss = recon + lam_l1 * l1 + lam_loc * locality

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

    with torch.no_grad():
        sim = m.similarity(model).item()
        acc = (m(test.x).argmax(-1) == test.y).float().mean().item()
        all_w = torch.cat([m.left.flatten(), m.right.flatten(), m.down.flatten()]).abs()
        sparsity = (all_w < 0.01).float().mean().item()
        Labs, Rabs = m.left.abs(), m.right.abs()
        num = torch.einsum('iH, ij, jH -> H', Labs, dist, Rabs)
        den = torch.einsum('iH, jH -> H', Labs, Rabs).clamp(min=1e-8)
        avg_dist = (num / den).mean().item()
    return m, dict(lam_loc=lam_loc, sim=sim, acc=acc, sparsity=sparsity, avg_dist=avg_dist)


lams_loc = [0.0, 0.5, 2.0, 10.0]
fixed_l1 = 2.0  # from Approach 1: clean sparsity at full accuracy

torch.set_grad_enabled(True)
loc_models = []
loc_results = []
for lam_loc in (pbar := tqdm(lams_loc)):
    m, r = train_sparse_l1_loc(fixed_l1, lam_loc)
    loc_models.append(m)
    loc_results.append(r)
    pbar.set_description(
        f"lam_loc={lam_loc}: sim={r['sim']:.3f}, acc={r['acc']:.3f}, "
        f"sparse={r['sparsity']:.2f}, dist={r['avg_dist']:.2f}"
    )
torch.set_grad_enabled(False)

df_loc = pd.DataFrame(loc_results)
print(df_loc.round(3).to_string(index=False))

lam_loc=10.0: sim=0.690, acc=0.610, sparse=0.86, dist=0.12: 100%|██████████| 4/4 [00:16<00:00,  4.19s/it]

 lam_loc   sim   acc  sparsity  avg_dist
     0.0 0.999 0.967     0.804     0.282
     0.5 0.984 0.967     0.764     0.208
     2.0 0.940 0.958     0.755     0.158
    10.0 0.690 0.610     0.859     0.116


In [62]:
fig = plot_component_grid(
    loc_models,
    [f"λloc={lam}" for lam in lams_loc],
    n_comp=8,
    height_per_group=360,
)
fig


### Approach 2: predictive sufficiency of weight components

The experiments above make the decompositions visually cleaner, but the project framing asks for a stronger criterion: **does your decomposition actually help you predict?** An interesting way to test this is a top-$k$ sufficiency curve.

For each weight-only decomposition, sort components by their scale $\sigma_r = \|L_r\|\,\|R_r\|\,\|D_r\|$, keep only the top $k$, and evaluate two quantities:

$$
B^{(k)}_{c,i,j} = \sum_{r \in \text{top-}k} D_{c,r} L_{i,r} R_{j,r}.
$$

- **Tensor fidelity:** cosine similarity between $B^{(k)}$ and the original model's interaction tensor.
- **Held-out predictive sufficiency:** MNIST test accuracy using only those $k$ components.

The important distinction is that the components are still learned from the model weights; the dataset is only used after the fact as a behavioral validation set.


In [63]:
def component_order(m):
    sigma = m.left.norm(dim=0) * m.right.norm(dim=0) * m.down.norm(dim=0)
    return sigma.argsort(descending=True), sigma


def logits_topk(m, x, k):
    idx, _ = component_order(m)
    idx = idx[:k]
    x = x.flatten(start_dim=1)
    left, right, down = m.left[:, idx], m.right[:, idx], m.down[:, idx]
    return ((x @ left) * (x @ right)) @ down.T


def original_interaction_tensor(model):
    l, r = model.w_lr[0].unbind()
    l, r = l @ model.w_e, r @ model.w_e
    b = torch.einsum('co,oi,oj->cij', model.w_u, l, r)
    return 0.5 * (b + b.mT)


def tensor_topk(m, k):
    idx, _ = component_order(m)
    idx = idx[:k]
    b = torch.einsum('cr,ir,jr->cij', m.down[:, idx], m.left[:, idx], m.right[:, idx])
    return 0.5 * (b + b.mT)


def tensor_cosine(a, b):
    return (a * b).sum() / (a.norm() * b.norm()).clamp(min=1e-8)


def sparsity_topk(m, k, eps=1e-2):
    idx, _ = component_order(m)
    idx = idx[:k]
    weights = torch.cat([m.left[:, idx].flatten(), m.right[:, idx].flatten(), m.down[:, idx].flatten()])
    return (weights.abs() < eps).float().mean().item()


def locality_topk(m, k):
    idx, _ = component_order(m)
    idx = idx[:k]
    p = int(m.left.shape[0] ** 0.5)
    coords = torch.stack(torch.meshgrid(
        torch.arange(p, dtype=torch.float, device=m.left.device),
        torch.arange(p, dtype=torch.float, device=m.left.device),
        indexing='ij',
    ), -1).reshape(-1, 2)
    pair_dist = torch.cdist(coords, coords)
    pair_dist = pair_dist / pair_dist.max()

    labs, rabs = m.left[:, idx].abs(), m.right[:, idx].abs()
    num = torch.einsum('ir,ij,jr->r', labs, pair_dist, rabs)
    den = torch.einsum('ir,jr->r', labs, rabs).clamp(min=1e-8)
    return (num / den).mean().item()


pareto_models = {
    'CPD': sparse,
    'L1 λ=2': sparsity_models[lams.index(2.0)],
    'D-sparse λ=5': d_models[lams_d.index(5.0)],
    'L1+local λ=0.5': loc_models[lams_loc.index(0.5)],
    'L1+local λ=2': loc_models[lams_loc.index(2.0)],
}

ks = [1, 2, 4, 8, 16, 32, 64]
rows = []

with torch.no_grad():
    target = original_interaction_tensor(model)
    for name, m in pareto_models.items():
        for k in ks:
            logits = logits_topk(m, test.x, k)
            rows.append(dict(
                model=name,
                k=k,
                acc=(logits.argmax(-1) == test.y).float().mean().item(),
                sim=tensor_cosine(target, tensor_topk(m, k)).item(),
                sparsity=sparsity_topk(m, k),
                avg_dist=locality_topk(m, k),
            ))

df_pareto = pd.DataFrame(rows)
print('Held-out accuracy by top-k components')
print(df_pareto.pivot(index='k', columns='model', values='acc').round(3).to_string())
print('\nTensor cosine by top-k components')
print(df_pareto.pivot(index='k', columns='model', values='sim').round(3).to_string())


Held-out accuracy by top-k components
model    CPD  D-sparse λ=5  L1 λ=2  L1+local λ=0.5  L1+local λ=2
k                                                               
1      0.145         0.107   0.160           0.087         0.171
2      0.270         0.169   0.120           0.095         0.283
4      0.308         0.171   0.086           0.070         0.360
8      0.459         0.241   0.133           0.117         0.519
16     0.577         0.367   0.215           0.156         0.744
32     0.789         0.720   0.438           0.567         0.933
64     0.967         0.967   0.967           0.967         0.958

Tensor cosine by top-k components
model    CPD  D-sparse λ=5  L1 λ=2  L1+local λ=0.5  L1+local λ=2
k                                                               
1      0.226         0.095   0.144           0.034         0.274
2      0.338         0.212   0.099           0.092         0.335
4      0.469         0.298   0.190           0.150         0.420
8      0.610     

In [66]:
# Pareto plots for the top-k component evaluation.
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Held-out Accuracy', 'Tensor Cosine'],
    horizontal_spacing=0.12,
)

for name, group in df_pareto.groupby('model'):
    group = group.sort_values('k')
    fig.add_scatter(x=group.k, y=group.acc, name=name, mode='lines+markers', row=1, col=1)
    fig.add_scatter(x=group.k, y=group.sim, name=name, mode='lines+markers', showlegend=False, row=1, col=2)

fig.update_xaxes(type='log', tickvals=ks, title='top-k components')
fig.update_yaxes(range=[0, 1.02])
fig.update_layout(width=900, height=360, template='plotly_white', margin=dict(l=40, r=20, b=40, t=40))
fig


In [68]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Factor Sparsity', 'Average Pixel-Pair Distance'],
    horizontal_spacing=0.12,
)

for name, group in df_pareto.groupby('model'):
    group = group.sort_values('k')
    fig.add_scatter(x=group.k, y=group.sparsity, name=name, mode='lines+markers', row=1, col=1)
    fig.add_scatter(x=group.k, y=group.avg_dist, name=name, mode='lines+markers', showlegend=False, row=1, col=2)

fig.update_xaxes(type='log', tickvals=ks, title='top-k components')
fig.update_yaxes(range=[0, 1.02], row=1, col=1)
fig.update_yaxes(range=[0, 0.5], row=1, col=2)
fig.update_layout(width=900, height=360, template='plotly_white', margin=dict(l=40, r=20, b=40, t=40))
fig
